# Museums and Nature Centers — Solution

**Goal:** Apply intermediate ggplot2 techniques (histograms, box plots, stacked/fill bars, facets, scales, error bars) to explore the geographic distribution, type mix, and revenue profile of U.S. museums, aquariums, zoos and nature centers (2013 administrative + IRS data).

![Workflow flowchart](museums_nature_centers_flowchart.png)

---
## Setup

In [ ]:
library(dplyr)
library(ggplot2)
library(stringr)
library(tidyr)
library(plotrix)   # std.error()
library(scales)

## Part 1 — Data Exploration

In [ ]:
# Task 1–2
museums_df <- read.csv("data/museums.csv", stringsAsFactors = FALSE)

head(museums_df)
glimpse(museums_df)          # or str(museums_df)
summary(museums_df$Annual.Revenue)
table(museums_df$Is.Museum)
table(museums_df$Region.Code..AAM.)

## Part 2 — Explore Institutions by Type

In [ ]:
# Tasks 3–4 — bar of Museum.Type with wrapped labels
museum_type <- ggplot(museums_df, aes(x = Museum.Type)) +
  geom_bar(fill = "#2980b9") +
  scale_x_discrete(labels = scales::wrap_format(8)) +
  labs(title = "Institutions by Detailed Type",
       x = "Museum Type", y = "Count") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 8))

museum_type
# Historic Preservation is by far the most common category.

In [ ]:
# Task 5 — Museum vs Non-Museum
museum_class <- ggplot(museums_df, aes(x = Is.Museum)) +
  geom_bar(fill = c("#e74c3c", "#3498db")) +
  scale_x_discrete(labels = c("TRUE" = "Museum", "FALSE" = "Non-Museum",
                              "True" = "Museum", "False" = "Non-Museum")) +
  labs(title = "Classic Museums vs Zoos / Nature Centers / Historic Sites",
       x = "Category", y = "Count") +
  theme_minimal()

museum_class

In [ ]:
# Task 6 — filter states + facet
museums_states <- museums_df %>%
  filter(State..Administrative.Location. %in% c("IL", "CA", "NY"))

museum_facet <- ggplot(museums_states, aes(x = Is.Museum)) +
  geom_bar(fill = "#2c7bb6") +
  scale_x_discrete(labels = c("TRUE" = "Museum", "FALSE" = "Non-Museum",
                              "True" = "Museum", "False" = "Non-Museum")) +
  facet_grid(cols = vars(State..Administrative.Location.)) +
  labs(title = "Museum vs Non-Museum by Selected States",
       x = "Category", y = "Count") +
  theme_minimal()

museum_facet

In [ ]:
# Tasks 7–10 — stacked / fill by region
region_labels <- c("1" = "New England", "2" = "Mid-Atlantic",
                   "3" = "Southeastern", "4" = "Midwest",
                   "5" = "Mountain Plains", "6" = "Western")

museum_stacked <- ggplot(museums_df,
                         aes(x = factor(Region.Code..AAM.),
                             fill = factor(Is.Museum))) +
  geom_bar(position = "fill") +
  scale_x_discrete(labels = region_labels) +
  scale_y_continuous(labels = scales::percent_format()) +
  scale_fill_discrete(labels = c("TRUE" = "Museum", "FALSE" = "Non-Museum",
                                 "True" = "Museum", "False" = "Non-Museum")) +
  labs(title = "Museum Types by Region",
       x = "Region", y = "Percentage of Total", fill = "Type") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

museum_stacked
# Western and Mountain Plains tend to have a higher share of classic museums;
# Midwest has a higher share of Non-Museum institutions.

## Part 3 — Explore Institutions by Revenue

In [ ]:
# Task 11 — revenue subsets (parent-level, positive revenue)
museums_revenue_df <- museums_df %>%
  distinct(Legal.Name, .keep_all = TRUE) %>%
  filter(Annual.Revenue > 0)

museums_small_df <- museums_revenue_df %>%
  filter(Annual.Revenue < 1e6)

museums_large_df <- museums_revenue_df %>%
  filter(Annual.Revenue > 1e9)

nrow(museums_revenue_df)
nrow(museums_small_df)
nrow(museums_large_df)

In [ ]:
# Tasks 12–13 — histogram of small-museum revenue
revenue_histogram <- ggplot(museums_small_df, aes(x = Annual.Revenue)) +
  geom_histogram(binwidth = 1e4, fill = "#e67e22", colour = "white") +
  scale_x_continuous(labels = scales::dollar_format()) +
  labs(title = "Distribution of Annual Revenue — Small Institutions (< $1 M)",
       x = "Annual Revenue", y = "Count") +
  theme_minimal()

revenue_histogram

In [ ]:
# Tasks 14–16 — boxplot of large institutions by region
revenue_boxplot <- ggplot(museums_large_df,
                          aes(x = factor(Region.Code..AAM.),
                              y = Annual.Revenue)) +
  geom_boxplot(fill = "#9b59b6", alpha = 0.7) +
  coord_cartesian(ylim = c(1e9, 3e10)) +
  scale_y_continuous(labels = function(x) paste0("$", x / 1e9, "B")) +
  scale_x_discrete(labels = region_labels) +
  labs(title = "Large Institutions (> $1 B) — Revenue by Region",
       subtitle = "Y-axis zoomed; extreme outlier(s) still present",
       x = "Region", y = "Annual Revenue") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

revenue_boxplot

In [ ]:
# Tasks 17–18 — mean revenue bar
revenue_barplot <- ggplot(museums_revenue_df,
                          aes(x = factor(Region.Code..AAM.),
                              y = Annual.Revenue)) +
  geom_bar(stat = "summary", fun = "mean", fill = "#27ae60") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_discrete(labels = region_labels) +
  labs(title = "Mean Annual Revenue by Region",
       y = "Mean Annual Revenue", x = "Region") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

revenue_barplot

In [ ]:
# Task 19 — means + standard errors + error bars
museums_error_df <- museums_revenue_df %>%
  group_by(Region.Code..AAM.) %>%
  summarise(
    Mean.Revenue = mean(Annual.Revenue, na.rm = TRUE),
    Mean.SE      = std.error(Annual.Revenue),
    n            = n(),
    .groups = "drop"
  ) %>%
  mutate(
    SE.Min = Mean.Revenue - Mean.SE,
    SE.Max = Mean.Revenue + Mean.SE
  )

revenue_errorbar <- ggplot(museums_error_df,
                           aes(x = factor(Region.Code..AAM.),
                               y = Mean.Revenue)) +
  geom_col(fill = "#27ae60") +
  geom_errorbar(aes(ymin = SE.Min, ymax = SE.Max), width = 0.2, colour = "#1a1a2e") +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_discrete(labels = region_labels) +
  labs(title = "Mean Annual Revenue by Region (with SE)",
       y = "Mean Annual Revenue", x = "Region") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

revenue_errorbar
museums_error_df

## Alternate Approaches

In [ ]:
# Alternate 1 — pre-summarise + geom_col (same mean bar)
mean_by_region <- museums_revenue_df %>%
  group_by(Region = factor(Region.Code..AAM., labels = region_labels)) %>%
  summarise(Mean.Revenue = mean(Annual.Revenue), .groups = "drop")

ggplot(mean_by_region, aes(x = Region, y = Mean.Revenue)) +
  geom_col(fill = "#16a085") +
  scale_y_continuous(labels = dollar_format()) +
  labs(title = "Mean Annual Revenue by Region (pre-summarised)",
       x = "Region", y = "Mean Annual Revenue") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

In [ ]:
# Alternate 2 — facet_wrap instead of facet_grid
ggplot(museums_states, aes(x = Is.Museum)) +
  geom_bar(fill = "#2980b9") +
  scale_x_discrete(labels = c("TRUE" = "Museum", "FALSE" = "Non-Museum",
                              "True" = "Museum", "False" = "Non-Museum")) +
  facet_wrap(~ State..Administrative.Location., nrow = 1) +
  labs(title = "Museum vs Non-Museum (facet_wrap)") +
  theme_minimal()

In [ ]:
# Alternate 3 — custom palette + theme polish
ggplot(museums_df, aes(x = factor(Region.Code..AAM.), fill = factor(Is.Museum))) +
  geom_bar(position = "fill") +
  scale_x_discrete(labels = region_labels) +
  scale_y_continuous(labels = percent_format()) +
  scale_fill_manual(values = c("TRUE" = "#2c7bb6", "FALSE" = "#d7191c",
                               "True" = "#2c7bb6", "False" = "#d7191c"),
                    labels = c("TRUE" = "Museum", "FALSE" = "Non-Museum",
                               "True" = "Museum", "False" = "Non-Museum")) +
  labs(title = "Museum Types by Region — Custom Colours",
       x = "Region", y = "Share", fill = "Type") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1),
        legend.position = "bottom")

## More Practice

In [ ]:
# Heatmap-style count of Type × Region (top categories only for readability)
top_types <- museums_df %>%
  count(Museum.Type, sort = TRUE) %>%
  slice_head(n = 6) %>%
  pull(Museum.Type)

type_region_counts <- museums_df %>%
  filter(Museum.Type %in% top_types) %>%
  count(Museum.Type, Region = factor(Region.Code..AAM., labels = region_labels))

ggplot(type_region_counts, aes(x = Region, y = Museum.Type, fill = n)) +
  geom_tile(colour = "white") +
  scale_fill_gradient(low = "#f7fbff", high = "#08306b") +
  labs(title = "Institution Counts — Top Types × Region",
       x = "Region", y = "Type", fill = "Count") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

In [ ]:
# Restrict to Western region (code 6) and re-examine revenue shape
western_small <- museums_small_df %>%
  filter(Region.Code..AAM. == 6)

ggplot(western_small, aes(x = Annual.Revenue)) +
  geom_histogram(binwidth = 1e4, fill = "#e67e22", colour = "white") +
  scale_x_continuous(labels = dollar_format()) +
  labs(title = "Western Region — Small Institution Revenue",
       x = "Annual Revenue", y = "Count") +
  theme_minimal()

## Simulation / Sensitivity

In [ ]:
# ---- Interactive parameters ----
small_threshold <- 1e6
large_threshold <- 1e9
hist_binwidth   <- 25000
states_of_interest <- c("TX", "FL", "WA")
sample_frac     <- 0.5   # simulate a 50 % sample

set.seed(42)
sim_revenue <- museums_df %>%
  distinct(Legal.Name, .keep_all = TRUE) %>%
  filter(Annual.Revenue > 0) %>%
  slice_sample(prop = sample_frac)

sim_small <- sim_revenue %>% filter(Annual.Revenue < small_threshold)

# Histogram under new parameters
ggplot(sim_small, aes(x = Annual.Revenue)) +
  geom_histogram(binwidth = hist_binwidth, fill = "#e67e22", colour = "white") +
  scale_x_continuous(labels = dollar_format()) +
  labs(title = paste0("Simulated Small-Institution Histogram\n",
                      "threshold = $", format(small_threshold, scientific = FALSE),
                      " | binwidth = $", format(hist_binwidth, scientific = FALSE),
                      " | sample = ", sample_frac * 100, "%"),
       x = "Annual Revenue", y = "Count") +
  theme_minimal()

# Mean + SE under the reduced sample
sim_error <- sim_revenue %>%
  group_by(Region.Code..AAM.) %>%
  summarise(Mean.Revenue = mean(Annual.Revenue),
            Mean.SE = std.error(Annual.Revenue),
            .groups = "drop") %>%
  mutate(SE.Min = Mean.Revenue - Mean.SE,
         SE.Max = Mean.Revenue + Mean.SE)

ggplot(sim_error, aes(x = factor(Region.Code..AAM.), y = Mean.Revenue)) +
  geom_col(fill = "#27ae60") +
  geom_errorbar(aes(ymin = SE.Min, ymax = SE.Max), width = 0.2) +
  scale_y_continuous(labels = dollar_format()) +
  scale_x_discrete(labels = region_labels) +
  labs(title = paste0("Mean Revenue under ", sample_frac * 100, "% Sample"),
       subtitle = "Error bars widen as sample size decreases",
       x = "Region", y = "Mean Annual Revenue") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

## Audience Adaptation — Quick Reference

| Audience | Recommended view | Title style | Extra |
|----------|------------------|-------------|-------|
| Executive | Fill-bar % + mean bar with SE | One clear headline | Drop technical footnotes |
| Analyst | Box plot + histogram + error bars | Precise, method-aware | Keep SE / n in caption |
| Public / press | Simple counts or % stacked | Plain language, short labels | Explain “why it matters” |

See the Strategy Guide and the two audience PDFs for the full checklist (data literacy, subject knowledge, mixed audiences, cultural conventions for numbers and colour).

## Key Takeaways from the Solution

1. **Historic Preservation** dominates the count of institutions; classic “museums” are a minority overall.
2. Regional composition of Museum vs Non-Museum is not uniform — Midwest leans more Non-Museum, Western & Mountain Plains lean more Museum.
3. Revenue is heavily right-skewed; a handful of very large parent organisations dominate the mean. Always inspect both the bulk (histogram of small orgs) and the tail (boxplot of large orgs).
4. Error bars on regional means reveal that some regions (especially those with fewer large institutions) have much wider uncertainty.
5. Always de-duplicate on `Legal.Name` before analysing revenue; otherwise parent organisations with multiple sites inflate the totals.